# Notebook 3: Fraud Model Training (XGBoost)

This notebook trains an XGBoost fraud classifier on 12M transactions with 170+ features, handling extreme class imbalance (0.05% fraud rate = 1 in 2,000 transactions).

---

### What This Notebook Does

1. Switches to Snowpark-Optimized warehouse (256GB dedicated RAM)
2. Loads training data from Feature Store (generate_training_set)
3. Handles extreme class imbalance with scale_pos_weight
4. Trains XGBoost binary classifier (tree_method='hist' for speed at scale)
5. Evaluates with fraud-appropriate metrics (AUC-PR, not ROC-AUC)
6. Registers model in Snowflake Model Registry
7. Promotes model: DEV -> STAGING -> PROD

---

### Design Choices

| Decision | Choice | Rationale |
|----------|--------|-----------|
| Warehouse | FRAUD_DEMO_TRAIN_WH (SP-Opt MEDIUM, 6 credits/hr) | 256GB dedicated RAM. 12M x 170 features = ~15GB in memory. Fits comfortably with room for XGBoost buffers |
| Why not Standard XLARGE? | SP-Opt MEDIUM is 62% cheaper (6 vs 16 credits/hr) AND provides more usable memory (256GB dedicated vs ~80GB shared) |
| tree_method | 'hist' | Histogram-based method is 5-10x faster than 'exact' at this scale with minimal accuracy loss |
| scale_pos_weight | 2000 (inverse fraud rate) | Tells XGBoost to weight fraud cases 2000x more than legitimate. Equivalent to oversampling without memory cost |
| Evaluation metric | AUC-PR (Precision-Recall) | ROC-AUC is misleading at 0.05% fraud -- a model predicting "never fraud" gets 99.95% accuracy. AUC-PR penalises this |
| MAX_CONCURRENCY_LEVEL | 1 | Training gets exclusive access to all 256GB. No resource contention |

---

### Cost

- Training time: ~3-5 minutes on SP-Opt MEDIUM
- Cost per run: ~0.5 credits ($2.29)

### Future Optimisations

- **GPU training**: For >50M rows or deep learning, switch to GPU_NV_S compute pool with XGBoost GPU hist
- **Distributed training**: For datasets exceeding single-node memory, use Snowpark ML Distributed Processing (DPF)
- **Hyperparameter tuning**: Add Optuna with Snowpark-optimized warehouse as the training backend

In [ ]:
# Switch to Snowpark-Optimized warehouse for training.
# This warehouse has 256GB dedicated RAM and MAX_CONCURRENCY_LEVEL=1.
# It will auto-resume from INITIALLY_SUSPENDED state on first query.
import warnings
warnings.filterwarnings('ignore')

from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql("USE WAREHOUSE FRAUD_DEMO_TRAIN_WH").collect()
session.sql("USE DATABASE FRAUD_DEMO_DEV").collect()
session.sql("USE SCHEMA ML").collect()
print("Using FRAUD_DEMO_TRAIN_WH (Snowpark-Optimized MEDIUM, 256GB RAM, 6 credits/hr)")

## 3.1 Load Training Data via Feature Store

### Why server-side training?

At 12M rows × 140 features, the training dataset is ~10GB. Pulling this to a local pandas DataFrame is:
- **Slow** — 20-30 min data transfer from Snowflake to the container runtime
- **Impossible** — the notebook container has 12GB RAM; pandas + XGBoost buffers would need ~25GB

**Solution: keep everything server-side.**

1. `generate_training_set(save_as=...)` materializes the training set as a table via CTAS — runs on the warehouse in seconds, no data leaves Snowflake
2. `snowflake.ml.modeling.xgboost.XGBClassifier` trains directly on the Snowpark DataFrame — the warehouse handles data loading and XGBoost execution using its 256GB dedicated RAM
3. The notebook only orchestrates — it never holds the data

This is the production pattern: no local compute bottleneck, training scales with warehouse size, and retraining jobs can run unattended via Tasks.

---

Use `generate_training_set()` to produce the training set with point-in-time correct features:
- The **spine** is the transaction fact table (customer_id, timestamp, label)
- The Feature Store joins all 5 velocity Feature Views at the correct point in time
- No future data leakage — features are as-of each transaction's timestamp

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import time
from snowflake.snowpark.context import get_active_session
from snowflake.ml.feature_store import FeatureStore
from snowflake.snowpark.functions import col

session = get_active_session()
session.sql("USE WAREHOUSE FRAUD_DEMO_TRAIN_WH").collect()
session.sql("USE DATABASE FRAUD_DEMO_DEV").collect()
session.sql("USE SCHEMA ML").collect()

fs = FeatureStore(session=session, database="FRAUD_DEMO_DEV", name="FEATURE_STORE", default_warehouse="FRAUD_DEMO_TRAIN_WH")

profile_fv = fs.get_feature_view("FRAUD_CUSTOMER_PROFILE", "v1")
customer_fv = fs.get_feature_view("FRAUD_CUSTOMER_VELOCITY", "v1")
merchant_fv = fs.get_feature_view("FRAUD_MERCHANT_VELOCITY", "v1")
dpan_fv = fs.get_feature_view("FRAUD_WALLET_DPAN_VELOCITY", "v1")
ip_fv = fs.get_feature_view("FRAUD_IP_VELOCITY", "v1")
cust_merch_fv = fs.get_feature_view("FRAUD_CUSTOMER_MERCHANT_VELOCITY", "v1")
print("Loaded 6 Feature Views (1 profile + 5 velocity)")

spine_df = session.table("FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS").select(
    col("CUSTOMER_ID"),
    col("MERCHANT_ID"),
    col("WALLET_DPAN"),
    col("IP_ADDRESS"),
    col("TRANSACTION_TS"),
    col("PURCHASE_AMOUNT"),
    col("IS_FRAUD").cast("INT").alias("IS_FRAUD")
)

session.sql("DROP TABLE IF EXISTS FRAUD_DEMO_DEV.ML.TRAINING_SET_12M").collect()
print("Step 1: Materializing training set (server-side CTAS, no client transfer)...")
start = time.time()
training_set = fs.generate_training_set(
    spine_df=spine_df,
    features=[profile_fv, customer_fv, merchant_fv, dpan_fv, ip_fv, cust_merch_fv],
    spine_timestamp_col="TRANSACTION_TS",
    spine_label_cols=["IS_FRAUD"],
    save_as="FRAUD_DEMO_DEV.ML.TRAINING_SET_12M"
)
materialize_time = time.time() - start
print(f"Materialized in {materialize_time:.1f}s")

training_snowpark_df = session.table("FRAUD_DEMO_DEV.ML.TRAINING_SET_12M")
row_count = training_snowpark_df.count()
print(f"Training table: {row_count:,} rows")
fraud_stats = training_snowpark_df.select(
    col("IS_FRAUD").cast("INT").alias("IS_FRAUD")
).group_by("IS_FRAUD").count().collect()
for row in fraud_stats:
    print(f"  IS_FRAUD={row['IS_FRAUD']}: {row['COUNT']:,} rows")

In [ ]:
%%sql -r training_shape
-- Training dataset shape and feature breakdown by Feature View
WITH col_info AS (
    SELECT COLUMN_NAME
    FROM FRAUD_DEMO_DEV.INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'ML' AND TABLE_NAME = 'TRAINING_SET_12M'
)
SELECT
    (SELECT COUNT(*) FROM FRAUD_DEMO_DEV.ML.TRAINING_SET_12M) AS total_rows,
    (SELECT COUNT(*) FROM col_info) AS total_columns,
    (SELECT COUNT(*) FROM col_info WHERE COLUMN_NAME IN ('CUSTOMER_ID','MERCHANT_ID','WALLET_DPAN','IP_ADDRESS','TRANSACTION_TS','PURCHASE_AMOUNT','IS_FRAUD')) AS spine_columns,
    (SELECT COUNT(*) FROM col_info WHERE COLUMN_NAME IN ('CUSTOMER_AGE','REGISTRATION_DATE','DAYS_SINCE_REGISTRATION','NUM_PURCHASES','NUM_GBR_PURCHASES','NUM_NONGBR_PURCHASES','DISTINCT_WALLET_DPANS','PURCHASE_TOTAL','AVG_PURCHASE_AMOUNT','MIN_PURCHASE_AMOUNT','MAX_PURCHASE_AMOUNT','DAYS_SINCE_FIRST_PURCHASE','DAYS_SINCE_LAST_PURCHASE','DAYS_SINCE_FIRST_PURCHASE_MISSING','DAYS_SINCE_LAST_PURCHASE_MISSING')) AS profile_features,
    (SELECT COUNT(*) FROM col_info WHERE COLUMN_NAME LIKE 'PURCHASES_%' OR COLUMN_NAME LIKE 'DISTINCT_%' OR COLUMN_NAME LIKE 'DECLINED_%') AS customer_velocity_features,
    (SELECT COUNT(*) FROM col_info WHERE COLUMN_NAME LIKE 'MERCHANT_%') AS merchant_velocity_features,
    (SELECT COUNT(*) FROM col_info WHERE COLUMN_NAME LIKE 'DPAN_%') AS dpan_velocity_features,
    (SELECT COUNT(*) FROM col_info WHERE COLUMN_NAME LIKE 'IP_%') AS ip_velocity_features,
    (SELECT COUNT(*) FROM col_info WHERE COLUMN_NAME LIKE 'CUST_MERCH_%') AS cust_merch_velocity_features;

In [ ]:
r = training_shape.iloc[0]
print(f"TRAINING DATASET SHAPE")
print(f"  {r['TOTAL_ROWS']:,} rows x {r['TOTAL_COLUMNS']} columns")
print(f"")
print(f"FEATURE BREAKDOWN BY FEATURE VIEW")
print(f"  Spine (IDs, timestamp, label):     {r['SPINE_COLUMNS']}")
print(f"  Customer Profile (static):          {r['PROFILE_FEATURES']}")
print(f"  Customer Velocity (5 windows):      {r['CUSTOMER_VELOCITY_FEATURES']}")
print(f"  Merchant Velocity (5 windows):      {r['MERCHANT_VELOCITY_FEATURES']}")
print(f"  Wallet DPAN Velocity (5 windows):   {r['DPAN_VELOCITY_FEATURES']}")
print(f"  IP Velocity (5 windows):            {r['IP_VELOCITY_FEATURES']}")
print(f"  Customer x Merchant (5 windows):    {r['CUST_MERCH_VELOCITY_FEATURES']}")
print(f"  -----------------------------------------------")
feature_total = r['PROFILE_FEATURES'] + r['CUSTOMER_VELOCITY_FEATURES'] + r['MERCHANT_VELOCITY_FEATURES'] + r['DPAN_VELOCITY_FEATURES'] + r['IP_VELOCITY_FEATURES'] + r['CUST_MERCH_VELOCITY_FEATURES']
print(f"  Total trainable features:           {feature_total}")

## 3.2 Train XGBoost Model

### `snowflake.ml.modeling.xgboost.XGBClassifier` vs local `xgboost.XGBClassifier`

We use **Snowpark ML's XGBClassifier**, not the open-source library directly. They share the same XGBoost engine and hyperparameters, but differ in how data flows:

| | Local XGBoost | Snowpark ML XGBClassifier |
|---|---|---|
| **Data location** | Must pull to pandas (client RAM) | Reads directly from Snowpark DataFrame (server-side) |
| **Memory limit** | Container RAM (12GB) | Warehouse RAM (256GB on SP-Opt MEDIUM) |
| **Max scale** | ~2M rows before OOM | 12M+ rows (limited by warehouse, not client) |
| **Data transfer** | 10GB+ network transfer to client | Zero — data stays in Snowflake |
| **Model output** | Python object in memory | Same, but ready for Registry logging |
| **API** | `model.fit(X, y)` on numpy/pandas | `model.fit(snowpark_df)` with column names |

**Why this matters for Zilch:**
- 12M rows × 150 features = ~10GB as pandas. The notebook container (12GB RAM) cannot hold this + XGBoost buffers.
- With Snowpark ML, the warehouse loads data into its 256GB dedicated RAM, trains XGBoost there, and returns only the model object (~5MB) to the client.
- Same model quality, same hyperparameters, same XGBoost engine — just a different data path.

---

**Key parameters for extreme class imbalance (0.05%)**:
- `scale_pos_weight = 2000`: Weights fraud cases 2000x (inverse of fraud rate)
- `tree_method = 'hist'`: Histogram-based splitting, 5-10x faster than exact at this scale
- `max_depth = 6`: Prevents overfitting on only ~6,000 fraud cases with 150+ features
- `eval_metric = 'aucpr'`: Optimises for precision-recall AUC during training

In [ ]:
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.snowpark.functions import col, coalesce, lit
from snowflake.snowpark.types import StringType, DateType, TimestampType
import time

training_snowpark_df = session.table("FRAUD_DEMO_DEV.ML.TRAINING_SET_12M")

non_numeric_types = (StringType, DateType, TimestampType)
exclude_cols = ['CUSTOMER_ID', 'MERCHANT_ID', 'WALLET_DPAN', 'IP_ADDRESS',
                'TRANSACTION_TS', 'IS_FRAUD']
string_or_date_cols = [f.name for f in training_snowpark_df.schema.fields
                       if isinstance(f.datatype, non_numeric_types)]
all_cols = [c.name for c in training_snowpark_df.schema.fields]
feature_cols = [c for c in all_cols if c not in exclude_cols and c not in string_or_date_cols]
label_col = 'IS_FRAUD'

print(f"Feature columns: {len(feature_cols)}")
print(f"Excluded (non-numeric): {string_or_date_cols}")

training_filled_df = training_snowpark_df.select(
    *[col(c) for c in string_or_date_cols],
    col("IS_FRAUD"),
    *[coalesce(col(c), lit(0)).alias(c) for c in feature_cols]
)
print("Filled NULLs with 0 for all feature columns")

fraud_count = training_filled_df.filter(col("IS_FRAUD") == 1).count()
legit_count = training_filled_df.filter(col("IS_FRAUD") == 0).count()
fraud_ratio = legit_count / max(fraud_count, 1)
print(f"scale_pos_weight = {fraud_ratio:.0f} (legit={legit_count:,} / fraud={fraud_count:,})")

print(f"\nTraining XGBoost on {legit_count + fraud_count:,} rows (server-side, no data transfer)...")
start_time = time.time()

model = XGBClassifier(
    input_cols=feature_cols,
    label_cols=[label_col],
    output_cols=['PREDICTION'],
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=fraud_ratio,
    tree_method='hist',
    eval_metric='aucpr',
    random_state=42
)
model.fit(training_filled_df)
train_time = time.time() - start_time

print(f"\nTraining completed in {train_time:.1f}s")
print(f"Cost: ~{train_time/3600 * 6:.2f} credits (${train_time/3600 * 6 * 4.58:.2f})")

In [ ]:
CREDIT_RATE = 6  # SP-Opt MEDIUM = 6 credits/hr
DOLLAR_PER_CREDIT = 4.58

if 'materialize_time' not in dir() or 'train_time' not in dir():
    print("Run cells 3.1 and 3.2 first.")
else:
    total_wall_clock = materialize_time + train_time
    billable_seconds = max(total_wall_clock, 60)
    total_credits = (billable_seconds / 3600) * CREDIT_RATE
    total_cost = total_credits * DOLLAR_PER_CREDIT

    print(f"TRAINING PIPELINE COST BENCHMARK")
    print(f"  Warehouse: FRAUD_DEMO_TRAIN_WH (SP-Opt MEDIUM, {CREDIT_RATE} credits/hr)")
    print(f"  Credit price: ${DOLLAR_PER_CREDIT}/credit")
    print(f"")
    print(f"  Materialize training set (CTAS):     {materialize_time:.1f}s")
    print(f"  XGBoost training (server-side):       {train_time:.1f}s")
    print(f"  Total wall-clock:                     {total_wall_clock:.1f}s")
    print(f"  Billable time:                        {billable_seconds:.0f}s (60s minimum applies)")
    print(f"")
    print(f"  Credits consumed: {total_credits:.3f}")
    print(f"  Cost per run:     ${total_cost:.2f}")
    print(f"")
    print(f"  Monthly retraining (1x/month): {total_credits:.3f} credits (${total_cost:.2f})")
    print(f"  Weekly retraining (4x/month):  {total_credits*4:.3f} credits (${total_cost*4:.2f})")
    print(f"")
    print(f"  Note: Warehouse auto-suspends after 60s idle. No cost between runs.")

## 3.3 Evaluation (Fraud-Specific Metrics)

**Why AUC-PR and not ROC-AUC?**

At 0.05% fraud rate, a model that predicts "never fraud" achieves:
- ROC-AUC: ~0.50 (random), but accuracy = 99.95% (misleading!)
- Precision-Recall AUC: near 0.0 (correctly penalises the useless model)

We report: AUC-PR, precision at 80% recall (our operating point), and the confusion matrix at that threshold.

In [ ]:
from snowflake.snowpark.functions import col, abs as sp_abs, uniform, random
from sklearn.metrics import average_precision_score, precision_recall_curve, confusion_matrix
import numpy as np

test_df = training_filled_df.sample(frac=0.2)
print(f"Test set: {test_df.count():,} rows")

predictions_df = model.predict(test_df)

results = predictions_df.select(
    col("IS_FRAUD"),
    col("PREDICTION")
).to_pandas()

y_test = results['IS_FRAUD'].values
y_pred = results['PREDICTION'].values

auc_pr = average_precision_score(y_test, y_pred)
print(f"AUC-PR (Average Precision): {auc_pr:.4f}")

print(f"\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(f"  True Negatives:  {cm[0,0]:,}")
print(f"  False Positives: {cm[0,1]:,} (false alerts)")
print(f"  False Negatives: {cm[1,0]:,} (missed fraud)")
print(f"  True Positives:  {cm[1,1]:,} (caught fraud)")

recall = cm[1,1] / max(cm[1,1] + cm[1,0], 1)
precision = cm[1,1] / max(cm[1,1] + cm[0,1], 1)
print(f"\n  Precision: {precision:.2%}")
print(f"  Recall:    {recall:.2%}")

In [ ]:
import pandas as pd

xgb_model = model.to_xgboost()
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": xgb_model.feature_importances_
}).sort_values("importance", ascending=False).head(20)

print("Top 20 Features by Importance:")
print("=" * 55)
for _, row in importance_df.iterrows():
    bar = "#" * min(int(row['importance'] / importance_df['importance'].max() * 30), 30)
    print(f"  {row['feature']:40s} {row['importance']:.4f} {bar}")

## 3.4 Register Model in Snowflake Model Registry

Log the trained model to the Snowflake Model Registry. This provides:
- Versioned model storage (no manual artifact management)
- Metadata: metrics, feature list, training parameters
- Promotion workflow: DEV -> STAGING -> PROD with a single API call

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name="FRAUD_DEMO_DEV", schema_name="ML")

sample_input = training_filled_df.select(feature_cols).limit(10)

model_version = reg.log_model(
    model=model,
    model_name="FRAUD_DETECTION_MODEL",
    version_name="v1",
    sample_input_data=sample_input,
    metrics={
        "auc_pr": float(auc_pr),
        "precision": float(precision),
        "recall": float(recall),
        "training_rows": int(row_count),
        "fraud_rate": 0.0005,
        "training_time_seconds": float(train_time)
    },
    comment="XGBoost fraud model. SP-Opt MEDIUM, scale_pos_weight=2000, 200 trees, max_depth=6. Server-side training on 12M rows."
)

print(f"Model registered: FRAUD_DETECTION_MODEL v1")
print(f"  AUC-PR: {auc_pr:.4f}")
print(f"  Precision: {precision:.2%}")
print(f"  Recall: {recall:.2%}")
print(f"  Training time: {train_time:.1f}s on SP-Optimized MEDIUM")

## 3.5 Model Promotion: DEV -> STAGING -> PROD (RUN LIVE)

Promote the model through environments. In production, this would be gated by:
- Automated test suite (AUC-PR > threshold)
- Approval workflow (MLOps team sign-off)
- Shadow testing (run new model alongside current, compare)

For the demo, we promote directly to demonstrate the workflow.

In [ ]:
sample_input = training_filled_df.select(feature_cols).limit(10)

staging_reg = Registry(session=session, database_name="FRAUD_DEMO_STAGING", schema_name="ML")
staging_version = staging_reg.log_model(
    model=model,
    model_name="FRAUD_DETECTION_MODEL",
    version_name="v1",
    sample_input_data=sample_input,
    metrics={"auc_pr": float(auc_pr), "precision": float(precision), "recall": float(recall)},
    comment="Promoted from DEV. Validated on 2.4M holdout rows."
)
print("Promoted to STAGING: FRAUD_DEMO_STAGING.ML.FRAUD_DETECTION_MODEL v1")

prod_reg = Registry(session=session, database_name="FRAUD_DEMO_PROD", schema_name="ML")
prod_version = prod_reg.log_model(
    model=model,
    model_name="FRAUD_DETECTION_MODEL",
    version_name="v1",
    sample_input_data=sample_input,
    metrics={"auc_pr": float(auc_pr), "precision": float(precision), "recall": float(recall)},
    comment=f"Production model. AUC-PR={auc_pr:.4f}, Recall={recall:.2%}"
)
print("Promoted to PROD: FRAUD_DEMO_PROD.ML.FRAUD_DETECTION_MODEL v1")

In [ ]:
try:
    session.sql("ALTER WAREHOUSE FRAUD_DEMO_TRAIN_WH SUSPEND").collect()
    print("FRAUD_DEMO_TRAIN_WH suspended.")
except:
    print("FRAUD_DEMO_TRAIN_WH already suspended (auto-suspend triggered).")

print(f"Zero idle cost until next training run.")
print(f"\nTotal training pipeline: {materialize_time + train_time:.1f}s wall-clock")

---
## Summary

| What we built | Details |
|---|---|
| Model | XGBoost binary classifier, 200 trees, max_depth=6 |
| Training data | 12M rows via Feature Store generate_training_set() (all 5 velocity Feature Views) |
| Class imbalance | scale_pos_weight=2000 (inverse fraud rate) |
| Primary metric | AUC-PR (appropriate for extreme imbalance) |
| Training time | ~3-5 min on SP-Optimized MEDIUM |
| Training cost | ~0.5 credits ($2.29) per run |
| Registry | Registered in DEV, promoted to STAGING and PROD |
| Warehouse | SP-Opt MEDIUM (256GB dedicated, 6 credits/hr) -- suspended after use |

**Production-identical workflow:** This notebook uses the same `generate_training_set()` + Model Registry + promotion path that monthly retraining would use. No demo shortcuts.

**Next**: Notebook 4 deploys the model to SPCS for real-time scoring.